In [ ]:
import os
import requests
import cdsapi
import geopandas as gpd
import pandas as pd
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

## CHANGE DIRECTORYYYY
OUTPUT_DIR = "../../01_data/1.2_raw_api/era5_hourly_temp"
os.makedirs(OUTPUT_DIR, exist_ok=True)

GEO_DIR = "../../01_data/1.3_raw_other/mapping"
os.makedirs(GEO_DIR, exist_ok=True)

CLEANED_DIR = "../../01_data/2_cleaned/weather"
os.makedirs(CLEANED_DIR, exist_ok=True)

In [ ]:
# Texas state boundary, for clipping the bbox download down to the actual state shape
TEXAS_GEOJSON_URL = "https://raw.githubusercontent.com/glynnbird/usstatesgeojson/master/texas.geojson"
texas_geojson_path = os.path.join(GEO_DIR, "texas_state_boundary.geojson")

if not os.path.exists(texas_geojson_path):
    resp = requests.get(TEXAS_GEOJSON_URL, timeout=30)
    resp.raise_for_status()
    with open(texas_geojson_path, "wb") as f:
        f.write(resp.content)

texas_gdf = gpd.read_file(texas_geojson_path).to_crs("EPSG:4326")
texas_boundary = texas_gdf.union_all()
texas_gdf.plot(edgecolor="black", facecolor="none")

In [ ]:
# Texas geo box
# North, West, South, East
AREA = [37, -107, 25.5, -93]

In [ ]:
from dotenv import load_dotenv

load_dotenv()
CDS_API_URL = os.environ["CDS_API_URL"]
CDS_API_KEY = os.environ["CDS_API_KEY"]  # https://cds.climate.copernicus.eu/profile
client = cdsapi.Client(url=CDS_API_URL, key=CDS_API_KEY)

In [ ]:
# test cell
out_nc = os.path.join(OUTPUT_DIR,f"era5_texas_t2m_hourly_test.nc")
client.retrieve(
        "reanalysis-era5-single-levels",
        {
            "product_type": ["reanalysis"],
            "variable": ["2m_temperature"],
            "year": [str(2026)],
            "month": ['06'],
            "day": [f"{day:02d}" for day in range(1, 32)],
            "time": [f"{hour:02d}:00" for hour in range(24)],
            "area": AREA,
            "data_format": "netcdf",
            "download_format": "unarchived",
        },
        out_nc,
    )

In [ ]:
# mask the downloaded box down to grid cells whose center falls inside Texas
# and take hourly average temperature
def get_texas_mean(year):
    nc_path = os.path.join(OUTPUT_DIR, f"era5_texas_t2m_hourly_{year}.nc")
    ds = xr.open_dataset(nc_path)

    lon2d, lat2d = np.meshgrid(ds["longitude"].values, ds["latitude"].values)
    grid_points = gpd.GeoDataFrame(
        geometry=gpd.points_from_xy(lon2d.ravel(), lat2d.ravel()),
        crs="EPSG:4326",
    )
    in_texas = grid_points.within(texas_boundary).values.reshape(lat2d.shape)

    texas_mask = xr.DataArray(
        in_texas,
        dims=("latitude", "longitude"),
        coords={"latitude": ds["latitude"], "longitude": ds["longitude"]},
    )
    
    ds_texas = ds.where(texas_mask)

    # take average
    t2m_hourly_mean = ds_texas["t2m"].mean(dim=["latitude", "longitude"])

    t2m_hourly_mean_df = t2m_hourly_mean.to_dataframe(name="t2m_mean_K").reset_index()
    t2m_hourly_mean_df["t2m_mean_F"] = round((t2m_hourly_mean_df["t2m_mean_K"] - 273.15) * 9 / 5 + 32, 0)

    df = t2m_hourly_mean_df.filter(['valid_time', 't2m_mean_F'])
    return df

In [ ]:
year = 'test'
df = get_texas_mean(year)
df

In [ ]:
df.to_csv(os.path.join(CLEANED_DIR, f'Texas Hourly Temperature Average test.csv'), index=False)

In [ ]:
# sanity check: shaded region should trace the Texas outline, not the bbox rectangle
nc_path = os.path.join(OUTPUT_DIR, f"era5_texas_t2m_hourly_test.nc")
ds = xr.open_dataset(nc_path)

lon2d, lat2d = np.meshgrid(ds["longitude"].values, ds["latitude"].values)
grid_points = gpd.GeoDataFrame(
        geometry=gpd.points_from_xy(lon2d.ravel(), lat2d.ravel()),
        crs="EPSG:4326",
    )
in_texas = grid_points.within(texas_boundary).values.reshape(lat2d.shape)

texas_mask = xr.DataArray(
        in_texas,
        dims=("latitude", "longitude"),
        coords={"latitude": ds["latitude"], "longitude": ds["longitude"]},
    )
    
ds_texas = ds.where(texas_mask)

fig, ax = plt.subplots(figsize=(6, 6))
ds_texas["t2m"].mean("valid_time").plot(ax=ax, x="longitude", y="latitude", cmap="coolwarm")
texas_gdf.boundary.plot(ax=ax, color="black", linewidth=1)
ax.set_title("Mean t2m (K), masked to Texas boundary")
plt.show()